In [3]:
import numpy as np

# ---------------------------------------------------------------------
# Max pooling: slide a small window over the image, like convolution --
# but instead of multiply-and-sum with learned numbers, it just keeps
# the LARGEST value in each window. No learnable parameters at all.
# ---------------------------------------------------------------------

image = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 0],
    [2, 1, 7, 8],
    [0, 3, 4, 2],
], dtype=float)

print("Image (4x4):")
print(image)

def maxpool_by_hand(image, window=2, stride=2):
    img_h, img_w = image.shape
    out_h = img_h // stride
    out_w = img_w // stride
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            row_start, col_start = i * stride, j * stride
            patch = image[row_start:row_start+window, col_start:col_start+window]
            output[i, j] = patch.max()   # <- the ONLY operation: keep the biggest number
            print(f"  window at rows[{row_start}:{row_start+window}], "
                  f"cols[{col_start}:{col_start+window}]:")
            print(f"  {patch.tolist()}  -> max = {output[i,j]}")

    return output

print("\nSliding a 2x2 window, stride 2 (non-overlapping):")
result = maxpool_by_hand(image, window=2, stride=2)

print("\nOutput (2x2):")
print(result)
print(f"\nShrunk from {image.shape} to {result.shape} -- exactly half in each dimension.")
print("No numbers were learned or multiplied. It's just 'look at 4 numbers, keep the biggest.'")


Image (4x4):
[[1. 3. 2. 4.]
 [5. 6. 1. 0.]
 [2. 1. 7. 8.]
 [0. 3. 4. 2.]]

Sliding a 2x2 window, stride 2 (non-overlapping):
  window at rows[0:2], cols[0:2]:
  [[1.0, 3.0], [5.0, 6.0]]  -> max = 6.0
  window at rows[0:2], cols[2:4]:
  [[2.0, 4.0], [1.0, 0.0]]  -> max = 4.0
  window at rows[2:4], cols[0:2]:
  [[2.0, 1.0], [0.0, 3.0]]  -> max = 3.0
  window at rows[2:4], cols[2:4]:
  [[7.0, 8.0], [4.0, 2.0]]  -> max = 8.0

Output (2x2):
[[6. 4.]
 [3. 8.]]

Shrunk from (4, 4) to (2, 2) -- exactly half in each dimension.
No numbers were learned or multiplied. It's just 'look at 4 numbers, keep the biggest.'


In [4]:
import numpy as np

def maxpool(image, window=2, stride=2):
    h, w = image.shape
    out_h, out_w = h // stride, w // stride
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            out[i, j] = image[i*stride:i*stride+window, j*stride:j*stride+window].max()
    return out

# ---------------------------------------------------------------------
# Clean test: ONE bright pixel (value 9) on a flat background (value 0).
# No convolution, no negative numbers, no wraparound -- just pooling
# itself, so there's only one thing being tested at a time.
# ---------------------------------------------------------------------

def make_spike(position, size=4):
    img = np.zeros((size, size))
    img[position] = 9
    return img

print("A 4x4 image is pooled with a 2x2 window, stride 2.")
print("That means the image splits into exactly 4 non-overlapping windows:")
print("  window A = rows[0:2], cols[0:2]  (top-left)")
print("  window B = rows[0:2], cols[2:4]  (top-right)")
print("  window C = rows[2:4], cols[0:2]  (bottom-left)")
print("  window D = rows[2:4], cols[2:4]  (bottom-right)")

positions_in_window_A = [(0, 0), (0, 1), (1, 0), (1, 1)]  # all 4 pixels inside window A

print("\n--- Moving the spike to different pixels, but ALL within window A ---\n")
for pos in positions_in_window_A:
    img = make_spike(pos)
    pooled = maxpool(img)
    print(f"Spike at pixel {pos}:")
    print(img)
    print("Pooled result:")
    print(pooled)
    print()

print("="*60)
print("Notice: the pooled output was IDENTICAL for all 4 positions above.")
print("Once you only have the pooled output, you cannot tell which of the")
print("4 pixels inside window A the spike was actually at -- that exact")
print("position information is gone. You only know 'somewhere in window A'.")
print("="*60)

print("\n--- Now move the spike OUTSIDE window A, into window B (one pixel over) ---\n")
img_A = make_spike((0, 1))   # last pixel still inside window A
img_B = make_spike((0, 2))   # one pixel further right -> now inside window B
pooled_A = maxpool(img_A)
pooled_B = maxpool(img_B)
print("Spike at (0,1) [inside window A], pooled result:")
print(pooled_A)
print("\nSpike at (0,2) [inside window B, just one pixel over], pooled result:")
print(pooled_B)
print("\nAre these two pooled outputs identical?", np.array_equal(pooled_A, pooled_B))
print("This time they're DIFFERENT -- crossing a window boundary DOES show up.")
print("So pooling doesn't destroy all position information -- it keeps which")
print("window something was in, but throws away exactly where within that window.")


A 4x4 image is pooled with a 2x2 window, stride 2.
That means the image splits into exactly 4 non-overlapping windows:
  window A = rows[0:2], cols[0:2]  (top-left)
  window B = rows[0:2], cols[2:4]  (top-right)
  window C = rows[2:4], cols[0:2]  (bottom-left)
  window D = rows[2:4], cols[2:4]  (bottom-right)

--- Moving the spike to different pixels, but ALL within window A ---

Spike at pixel (0, 0):
[[9. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Pooled result:
[[9. 0.]
 [0. 0.]]

Spike at pixel (0, 1):
[[0. 9. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Pooled result:
[[9. 0.]
 [0. 0.]]

Spike at pixel (1, 0):
[[0. 0. 0. 0.]
 [9. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Pooled result:
[[9. 0.]
 [0. 0.]]

Spike at pixel (1, 1):
[[0. 0. 0. 0.]
 [0. 9. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Pooled result:
[[9. 0.]
 [0. 0.]]

Notice: the pooled output was IDENTICAL for all 4 positions above.
Once you only have the pooled output, you cannot tell which of the
4 pixel

In [5]:
import numpy as np

kernel = np.array([
    [1, 0, -1],
    [0, 1,  0],
    [-1, 0, 1],
], dtype=float)

def convolve_by_hand(image, kernel):
    img_h, img_w = image.shape
    k_h, k_w = kernel.shape
    out_h, out_w = img_h - k_h + 1, img_w - k_w + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i+k_h, j:j+k_w]
            output[i, j] = np.sum(patch * kernel)
    return output

# --- Build clean, unambiguous test images: ONE edge orientation each ---
def vertical_edge():
    # bright on the left, dark on the right -- a clean VERTICAL edge
    img = np.zeros((7, 7))
    img[:, :3] = 1.0
    return img

def horizontal_edge():
    # bright on top, dark on bottom -- a clean HORIZONTAL edge
    img = np.zeros((7, 7))
    img[:3, :] = 1.0
    return img

def diagonal_edge_main():
    # bright top-left triangle, dark bottom-right -- diagonal edge along the MAIN diagonal
    img = np.zeros((7, 7))
    for i in range(7):
        for j in range(7):
            if i + j < 6:
                img[i, j] = 1.0
    return img

def diagonal_edge_anti():
    # bright bottom-left triangle, dark top-right -- diagonal edge along the ANTI-diagonal
    img = np.zeros((7, 7))
    for i in range(7):
        for j in range(7):
            if i > j:
                img[i, j] = 1.0
    return img

test_images = {
    "Vertical edge":         vertical_edge(),
    "Horizontal edge":       horizontal_edge(),
    "Diagonal edge (main)":  diagonal_edge_main(),
    "Diagonal edge (anti)":  diagonal_edge_anti(),
}

print("Kernel being tested:")
print(kernel)
print()

for name, img in test_images.items():
    response = convolve_by_hand(img, kernel)
    # Use the strongest single response as a summary number
    max_abs_response = np.abs(response).max()
    mean_abs_response = np.abs(response).mean()
    print(f"{name:22s}  max|response| = {max_abs_response:.2f}   mean|response| = {mean_abs_response:.3f}")


Kernel being tested:
[[ 1.  0. -1.]
 [ 0.  1.  0.]
 [-1.  0.  1.]]

Vertical edge           max|response| = 1.00   mean|response| = 0.400
Horizontal edge         max|response| = 1.00   mean|response| = 0.400
Diagonal edge (main)    max|response| = 1.00   mean|response| = 0.480
Diagonal edge (anti)    max|response| = 2.00   mean|response| = 1.040


In [6]:
import numpy as np

sobel_v = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=float)

# ---------------------------------------------------------------------
# KEY INSIGHT: the Sobel kernel is actually two simple 1D operations
# multiplied together (an "outer product"). Let's prove that first.
# ---------------------------------------------------------------------
derivative_1d = np.array([-1, 0, 1])   # "how much did the value change, left to right?"
smoothing_1d = np.array([1, 2, 1])     # "average with neighbors above/below" (noise reduction)

reconstructed = np.outer(smoothing_1d, derivative_1d)
print("smoothing_1d (column) x derivative_1d (row):")
print(reconstructed)

print("\nActual Sobel-vertical kernel:")
print(sobel_v)
print("\nAre they identical?", np.array_equal(reconstructed, sobel_v))

print("\n" + "="*70)
print("So Sobel-vertical = [a horizontal derivative] blended with")
print("                     [a vertical smoothing], done at the same time.")
print("="*70)

# ---------------------------------------------------------------------
# Let's isolate what EACH piece does, one at a time
# ---------------------------------------------------------------------
print("\nPart A: derivative_1d ALONE (as a 1x3 kernel) on a simple 1D signal")
signal = np.array([5, 5, 5, 1, 1, 1], dtype=float)  # flat, then a sudden DROP
print("Signal:", signal)
out = np.convolve(signal, derivative_1d[::-1], mode="valid")  # flipped for true convolution
print("Response:", out, " <- near-zero where flat, large where the value CHANGES")

# ---------------------------------------------------------------------
# Part B: apply full Sobel-vertical to clean test edges (same test images
# used for the diagonal kernel, so results are directly comparable)
# ---------------------------------------------------------------------
print("\nPart B: full Sobel-vertical kernel on clean test edges")
test_images = {
    "Vertical edge (left bright, right dark)":   vertical_edge(),
    "Horizontal edge (top bright, bottom dark)": horizontal_edge(),
    "Diagonal edge (main)":                      diagonal_edge_main(),
}
for name, img in test_images.items():
    response = convolve_by_hand(img, sobel_v)
    print(f"{name:45s} max|response| = {np.abs(response).max():.2f}   mean|response| = {np.abs(response).mean():.3f}")


smoothing_1d (column) x derivative_1d (row):
[[-1  0  1]
 [-2  0  2]
 [-1  0  1]]

Actual Sobel-vertical kernel:
[[-1.  0.  1.]
 [-2.  0.  2.]
 [-1.  0.  1.]]

Are they identical? True

So Sobel-vertical = [a horizontal derivative] blended with
                     [a vertical smoothing], done at the same time.

Part A: derivative_1d ALONE (as a 1x3 kernel) on a simple 1D signal
Signal: [5. 5. 5. 1. 1. 1.]
Response: [ 0. -4. -4.  0.]  <- near-zero where flat, large where the value CHANGES

Part B: full Sobel-vertical kernel on clean test edges
Vertical edge (left bright, right dark)       max|response| = 4.00   mean|response| = 1.600
Horizontal edge (top bright, bottom dark)     max|response| = 0.00   mean|response| = 0.000
Diagonal edge (main)                          max|response| = 3.00   mean|response| = 1.360
